
$\qquad$ $\qquad$$\qquad$  **TDA206/DIT206 Discrete Optimization: Assignment 1 -- Modelling and Solving LPs** <br />
$\qquad$ $\qquad$$\qquad$                   **Grader: N/A** <br />
$\qquad$ $\qquad$$\qquad$                     **Due Date: 27th January** <br />
$\qquad$ $\qquad$$\qquad$                   **Submitted by: Elias Söderberg, 030507-8792., elisod@chalmers.se** <br />


--- 


General guidelines:
*   All solutions to theoretical and pratical problems must be submitted in this ipynb notebook, and equations wherever required, should be formatted using LaTeX math-mode.
*   All discussion regarding practical problems, along with solutions and plots should be specified in this notebook. All plots/results should be visible such that the notebook do not have to be run. But the code in the notebook should reproduce the plots/results if we choose to do so.
*   Your name, personal number and email address should be specified above.
*   All tables and other additional information should be included in this notebook.
*   Before submitting, make sure that your Jupyter notebook run on another computer. That all plots can show on another computer including all your writing. It is good to check if your code can run here: https://colab.research.google.com.


# Problem 1.

Consider the following LP problem:


\begin{array}
\mathcal{max}\quad 4x_1-2x_2+5x_3+6x_4+7x_5\\
\textrm{s.t} \\
2x_1 + 2x_2 - 4x_3 + 4x_4 + 8x_5 \leq 6\\
2x_1 + x_2 - 2x_3 - x_4 - 3x_5 \geq -1\\
5x_1 - 2x_2 + 4x_3 + 4x_4 + 2x_5 = 5\\
2x_1 - 2x_2 + 5x_3 + 3x_4 + x_5 \leq 4\\
\vec x \geq \vec 0
\end{array}

* Use CVXPY to solve the LP above. Submit your code, and print the solution vector and objective value.

In [26]:
%pip install -q cvxpy
import cvxpy as cp

# later learned about n = 5, x = cp.Variable(n)
x1 = cp.Variable(nonneg=True)
x2 = cp.Variable(nonneg=True)
x3 = cp.Variable(nonneg=True)
x4 = cp.Variable(nonneg=True)
x5 = cp.Variable(nonneg=True)

constraints = [
    2*x1 + 2*x2 - 4*x3 + 4*x4 + 8*x5 <=  6,
    2*x1 + 1*x2 - 2*x3 - 1*x4 - 3*x5 >= -1,
    5*x1 - 2*x2 + 4*x3 + 4*x4 + 2*x5 ==  5,
    2*x1 - 2*x2 + 5*x3 + 3*x4 + 1*x5 <=  4,
]

obj = cp.Maximize(4*x1 -2*x2 + 5*x3 + 6*x4 + 7*x5)
prob = cp.Problem(obj, constraints)
prob.solve()
print("optimal value: ", prob.value)
print("solution vector:")
for i, x in enumerate([x1, x2, x3, x4, x5], start=1):
    print(f"x{i}: {x.value}")

Note: you may need to restart the kernel to use updated packages.
optimal value:  9.220338951278364
solution vector:
x1: 0.6271186572420929
x2: 2.813559402583746
x3: 1.54237290883075
x4: 1.174397310081799e-09
x5: 0.6610169394681931


# Problem 2.

There are 3 people who all start at the same location, and want to get to same end destination a total of $d=100$ units away. They have 1 bicycle, which they have to share. The amount of distance per unit time that the three people can cover walking and cycling is given below:

\begin{array}{l|c|c|} 
      & Walking Speed & Cycling Speed\\ \hline
 Person\ 1 &  1 & 6 \\
 Person\ 2 &  2 & 8 \\
 Person\ 3 &  1 & 6\\ \hline
\end{array}

Our goal is to minimize the time $t$ required for all people to reach the end destination. For each person consider 4 variables $x_i$, $u_i$, $y_i$, $z_i$, which respectively are the amount of time walking forwards, the amount of time walking backwards, the amount of time cycling forwards, and the amount of time cycling backwards. 

1. Formulate an LP to solve this problem for the generic where there are $n$ people trying to move a distance $d$, each with walking rate $w_i$ and biking rate $b_i$ for $i = 1\ldots n$.

    You should consider the following constraints
    - each person's total time for moving (forwards, backwards, walking, bike) not exceed the total time $t$
    - each person must make it all the way to $d$, recall that distance = velocity * time
    - the bike, riding forwards or backwards, can only be used for at most $t$ units of time.
    - the bike's distance to the start is at most $d$, remember that it can move both forwards and backwards.
    - time cannot be negative
   
2. Implement the generic LP in CVXPY and solve for the specific case of 3 people given above. Submit your code and write down objective and solution. Also confirm that the optimal solution satisfies $u_i = 0$ (i.e. no one ever walks backwards) and $y_iz_i = 0$ for all persons (i.e. no person ever cycles both forwards and backwards).

3. Use CVXPY to show what would happen if there was a 4th person who could walk at rate $3$ and cycle at rate $9$.


# Solution Part 1
The description uses the same names for parameters/variables described in the given problem, together with their original description

The objective function is to minimize $t$.

Each persons total moving time cannot exceed $t$:
$$
    x_i + u_i + y_i + z_i <= t, \quad \forall i = 1 ... n
$$

Each person must reach $d$:
$$
    w_i(x_i-u_i) + b_i(y_i-z_i) >= d, \quad \forall i = 1 ... n
$$

The bike can only be used for at most $t$ units of times:
$$
    \sum_{i=1}^{n}(y_i + z_i) <= t 
$$

The bike's distance to the start is at most $d$:
$$
    \sum_{i=1}^{n}(b_i y_i - b_i z_i) <= d 
$$

Time cannot be negative:
$$
    x_i, u_i, y_i, z_i, t >= 0, \quad \forall i = 1 ... n
$$


In [48]:
%pip install -q numpy
%pip install -q cvxpy
import numpy as np
import cvxpy as cp
from copy import deepcopy

def generic_problem(speeds: list[tuple[int, int]], distance: int):
    n = len(speeds)
    w = np.array([s[0] for s in speeds]) # walking speeds
    b = np.array([s[1] for s in speeds]) # biking speeds

    x = cp.Variable(n, nonneg=True) # forward
    u = cp.Variable(n, nonneg=True) # backward
    y = cp.Variable(n, nonneg=True) # bike forward
    z = cp.Variable(n, nonneg=True) # bike backward
    t = cp.Variable(nonneg=True)

    # non negative handled above
    constraints = []

    for i in range(n):
        # each person time <= total time
        constraints.append(x[i] + u[i] + y[i] + z[i] <= t)
        # each person travel >=d
        constraints.append(w[i]*(x[i]-u[i]) + b[i]*(y[i]-z[i]) >= d)

    # bike time <= t 
    constraints.append(cp.sum(y + z) <= t)

    # bike displacement <= d
    constraints.append(
        # mul - mul yields vec of everyone's distance; sum it for total; abs prob unnecessary 
        cp.abs(cp.sum(cp.multiply(b, y) - cp.multiply(b, z))) <= distance
    )

    obj = cp.Minimize(t)
    prob = cp.Problem(obj, constraints)
    prob.solve()
    
    precision = 5
    
    print(f"With {n} people:")
    print(f"Optimal time: {np.round(t.value, precision)}")

    
    for i, (xi, ui, yi, zi) in enumerate(zip(x, u, y, z), start=1):
        print(f"Person {i}:\n\t\
        Forward: {np.round(xi.value, precision)}\n\t\
        Backward: {np.round(ui.value, precision)}\n\t\
        Biking Forward: {np.round(yi.value, precision)}\n\t\
        Biking Backward: {np.round(zi.value, precision)}")    

three_speeds = [(1, 6), (2, 8), (1, 6)]
four_speeds = deepcopy(three_speeds)
four_speeds.append((3, 9))
d = 100
print("PART TWO:")
generic_problem(three_speeds, d)
print()
print("PART THREE:")
generic_problem(four_speeds, d)


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
PART TWO:
With 3 people:
Optimal time: 55.0
Person 1:
	        Forward: 46.0
	        Backward: 0.0
	        Biking Forward: 9.0
	        Biking Backward: 0.0
Person 2:
	        Forward: 54.0
	        Backward: 0.0
	        Biking Forward: 0.0
	        Biking Backward: 1.0
Person 3:
	        Forward: 46.0
	        Backward: 0.0
	        Biking Forward: 9.0
	        Biking Backward: 0.0

PART THREE:
With 4 people:
Optimal time: 47.6082
Person 1:
	        Forward: 37.12984
	        Backward: 0.0
	        Biking Forward: 10.47836
	        Biking Backward: 0.0
Person 2:
	        Forward: 46.81093
	        Backward: 0.0
	        Biking Forward: 0.79727
	        Biking Backward: 0.0
Person 3:
	        Forward: 37.12984
	        Backward: 0.0
	        Biking Forward: 10.47836
	        Biking Backward: 0.0
Person 4:
	        Forward: 44.03948
	        Backward: 0.

# Part 3 Discussion
Because of the fourth person's faster biking speed, they can help escort the bike backwards in order for the slower walkers to use it. This makes the fourth person a resource rather than a liability. The constraint that the bike cannot end up further than a total distance of $d$ from the starting position introduces a need for the bike to be shuttled back to slower riders.

# Question 3.
Use CVXPY to model and solve the other examples in the textbook (sections 2.1 -- 2.6).

In [83]:
%pip install -q numpy
%pip install -q cvxpy
import numpy as np
import cvxpy as cp

ROUND = 3
    
def p2_1():
    x = cp.Variable(3, nonneg=True)
    obj = cp.Minimize(0.75*x[0] + 0.5*x[1] + 0.15*x[2])
    constraints = [
        35*x[0] + 0.5*x[1] + 0.5*x[2] >= 0.5,
        60*x[0] + 300*x[1] +  10*x[2] >= 15,
        30*x[0] +  20*x[1] +  10*x[2] >= 4,
    ]

    prob = cp.Problem(obj, constraints)
    prob.solve()
    print(f"Optimal: {np.round(prob.value, ROUND)}")

def p2_2():
    x = cp.Variable(10)
    obj = cp.Maximize(x[0] + x[1] + x[2])
    constraints = [
        -3 <= x[0], x[0] <= 3,
        -1 <= x[1], x[1] <= 1,
        -1 <= x[2], x[2] <= 1,
        -1 <= x[3], x[3] <= 1,
        -1 <= x[4], x[4] <= 1,
        -3 <= x[5], x[5] <= 3,
        -4 <= x[6], x[6] <= 4,
        -4 <= x[7], x[7] <= 4,
        -4 <= x[8], x[8] <= 4,
        -1 <= x[9], x[9] <= 1,
        x[0] == x[3] + x[4],
        x[1] + x[3] == x[5],
        x[2] == x[6] + x[7],
        x[4] + x[6] == x[8],
        x[5] + x[7] == x[9],
    ]

    prob = cp.Problem(obj, constraints)
    prob.solve()
    print(f"Optimal: {np.round(prob.value, ROUND)}")

def p2_3():
    x = cp.Variable(13)
    y = cp.Variable(13)
    z = cp.Variable(13)
    s = cp.Variable(13)
    # eyeballing
    d = np.array([0, 350, 340, 450, 650, 650, 540, 680, 660, 330, 430, 390, 650])

    obj = cp.Minimize(50*cp.sum(y[1:]) + 50*cp.sum(z[1:]) + 20*cp.sum(s[1:]))
    constraints = [
        x[1:13] + s[0:12] - s[1:13] == d[1:13],
        x[1:13] - x[0:12] == y[1:13] - z[1:13],
        x[0] == 0,
        s[0] == 0,
        s[12] == 0,
        x[1:] >= 0,
        s[1:] >= 0,
        y[1:] >= 0,
        z[1:] >= 0,
    ]

    prob = cp.Problem(obj, constraints)
    prob.solve()
    print(f"Optimal: {np.round(prob.value, ROUND)}")

def p2_4(points: list[tuple[int, int]]):
    n = len(points)
    x = np.array([p[0] for p in points])
    y = np.array([p[1] for p in points])
    a = cp.Variable()
    b = cp.Variable()
    e = cp.Variable(n)

    obj = cp.Minimize(cp.sum(e))
    constraints = []
    for i in range(n):
        constraints.append(e[i] >= a*x[i] + b - y[i])
        constraints.append(e[i] >= -(a*x[i] + b - y[i]))

    prob = cp.Problem(obj, constraints)
    prob.solve()
    # plot points with plt if within leisure
    print(f"Optimal: {np.round(prob.value, ROUND)}")
    sym = "+ " if b.value >= 0 else ""
    print(f"Line given by: {np.round(a.value, ROUND)}x {sym}{np.round(b.value, ROUND)}")

def p2_5(m: list[tuple[int, int]], n: list[tuple[int, int]]):
    # case 1 left out because trivial (sort concatenated lists by x)
    # only handles case 2, where white points lie above and black points below
    # to handle case 3, switch the direction of inequalites >= / <= in for loops
    # this has been left out because trivial
    a = cp.Variable()
    b = cp.Variable()
    d = cp.Variable()

    obj = cp.Maximize(d)
    constraints = []
    for p in m:
        constraints.append(p[1] >= a*p[0] + b + d)

    for p in n:
        constraints.append(p[1] <= a*p[0] + b - d)

    prob = cp.Problem(obj, constraints)
    prob.solve()
    print(f"Optimal: {np.round(prob.value, ROUND)}")
    sym = "+ " if b.value >= 0 else ""
    print(f"Line given by: {np.round(a.value, ROUND)}x {sym}{np.round(b.value, ROUND)}")

def p2_6(lines: list[tuple[int, int]], k: int):
    # line y=ax+b in lines is (a, b)
    # lines are presumed to be sorted 
    # with k = cut-off described in book (1-indexed)

    n = len(lines)
    s1 = cp.Variable()
    s2 = cp.Variable()
    r = cp.Variable()

    obj = cp.Maximize(r)
    constraints = []
    
    for i in range(k):
        a = lines[i][0]
        b = lines[i][1]
        sq = np.sqrt(a**2 + 1)
        constraints.append(((s2-a * s1 - b)/(sq)) >= r)
    for i in range(k, n):
        a = lines[i][0]
        b = lines[i][1]
        sq = np.sqrt(a**2 + 1)
        constraints.append(((s2-a * s1 - b)/(sq)) <= -r)

    prob = cp.Problem(obj, constraints)
    prob.solve()
    print(f"Optimal r: {np.round(prob.value, ROUND)}")
    print(f"Center at: ({np.round(s1.value, ROUND)}, {np.round(s2.value, ROUND)})")

print("2.1:")
p2_1()

print("2.2:")
p2_2()

print("2.3:")
p2_3()

print("2.4:")
p2_4([(1, 2), (1, 3), (-1, 4)]) # made up points

print("2.5:")
p2_5([(2, 5), (3, 7)], [(1, 2), (5, 1)])

print("2.6:")
p2_6([(-1, 1), (1, -1), (0, 1)], 2)


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
2.1:
Optimal: 0.071
2.2:
Optimal: 4.0
2.3:
Optimal: 53366.665
2.4:
Optimal: 1.0
Line given by: -0.75x + 3.25
2.5:
Optimal: 1.625
Line given by: -0.25x + 3.875
2.6:
Optimal r: 0.414
Center at: (1.0, 0.586)
